# RunPod setup

Downloads two things this pipeline needs:

1. **Sample videos** → `data/` inside this repo (gitignored).
2. **SAM2 real-time fork + checkpoint** → `/workspace/segment-anything-2-real-time`
   (sibling of this repo, matching `sam2.repo_dir` in `src/config.yaml`).

Drive folder, clone URL, and checkpoint names come from the Roboflow notebook.
The pipeline only loads the **tiny** SAM2.1 checkpoint.

## Paths

On RunPod the persistent disk is `/workspace`. Keep this repo and SAM2 next to each other:

```
/workspace/
  Basketball-player-tracking/   ← this repo  (videos in data/)
  segment-anything-2-real-time/ ← SAM2 fork (cloned below)
```

In [1]:
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

WORKSPACE = Path("/workspace") if Path("/workspace").exists() else REPO_ROOT.parent
SOURCE_VIDEO_DIRECTORY = REPO_ROOT / "data"
SAM2_DIR = WORKSPACE / "segment-anything-2-real-time"

SOURCE_VIDEO_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("WORKSPACE:", WORKSPACE)
print("videos  →", SOURCE_VIDEO_DIRECTORY)
print("SAM2    →", SAM2_DIR)

REPO_ROOT: /workspace/Basketball-player-tracking
WORKSPACE: /workspace
videos  → /workspace/Basketball-player-tracking/data
SAM2    → /workspace/segment-anything-2-real-time


## 1. Sample videos

Celtics vs Knicks Game 1 clips from the [Roboflow notebook](https://github.com/roboflow-ai/notebooks). Skip this cell if `data/` already has the `.mp4` files.

In [2]:
!pip install -q gdown
!gdown -q https://drive.google.com/drive/folders/1eDJYqQ77Fytz15tKGdJCMeYSgmoQ-2-H -O {SOURCE_VIDEO_DIRECTORY} --folder

In [3]:
!ls -lh {SOURCE_VIDEO_DIRECTORY}

total 72M
-rw------- 1 root root 8.0M May 20  2025 boston-celtics-new-york-knicks-game-1-q1-01.54-01.48.mp4
-rw------- 1 root root 6.3M May 20  2025 boston-celtics-new-york-knicks-game-1-q1-03.16-03.11.mp4
-rw------- 1 root root  10M May 20  2025 boston-celtics-new-york-knicks-game-1-q1-04.28-04.20.mp4
-rw------- 1 root root 5.9M May 20  2025 boston-celtics-new-york-knicks-game-1-q1-04.44-04.39.mp4
-rw------- 1 root root 5.1M May 20  2025 boston-celtics-new-york-knicks-game-1-q1-05.13-05.09.mp4
-rw------- 1 root root 7.6M May 20  2025 boston-celtics-new-york-knicks-game-1-q1-06.00-05.54.mp4
-rw------- 1 root root 8.9M May 20  2025 boston-celtics-new-york-knicks-game-1-q1-07.41-07.34.mp4
-rw------- 1 root root 7.0M May 20  2025 boston-celtics-new-york-knicks-game-1-q2-08.09-08.03.mp4
-rw------- 1 root root 8.4M May 20  2025 boston-celtics-new-york-knicks-game-1-q2-08.43-08.38.mp4
-rw------- 1 root root 4.6M May 20  2025 boston-celtics-new-york-knicks-game-1-q2-10.36-10.32.mp4


## 2. SAM2 real-time

Clone [Gy920/segment-anything-2-real-time](https://github.com/Gy920/segment-anything-2-real-time) into `/workspace`.
`src/tracking.py` imports it from disk — no `pip install` needed.

Then download `sam2.1_hiera_tiny.pt` (the checkpoint in `src/config.yaml`).

In [4]:
if SAM2_DIR.exists():
    print("Already cloned:", SAM2_DIR)
else:
    !git clone https://github.com/Gy920/segment-anything-2-real-time.git {SAM2_DIR}

Cloning into '/workspace/segment-anything-2-real-time'...
remote: Enumerating objects: 406, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 406 (delta 65), reused 56 (delta 56), pack-reused 313 (from 2)
Receiving objects: 100% (406/406), 79.43 MiB | 38.79 MiB/s, done.
Resolving deltas: 100% (91/91), done.


In [5]:
CKPT = SAM2_DIR / "checkpoints" / "sam2.1_hiera_tiny.pt"
CKPT.parent.mkdir(parents=True, exist_ok=True)

if CKPT.exists():
    print("Already downloaded:", CKPT)
else:
    # Meta SAM2.1 weights (same URL as checkpoints/download_ckpts.sh in the fork)
    !wget -q -O {CKPT} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt

print(f"{CKPT.name}: {CKPT.stat().st_size / 1e6:.1f} MB")

# To pull every variant instead (large/base+/small/tiny), uncomment:
# !(cd {SAM2_DIR}/checkpoints && bash download_ckpts.sh)

sam2.1_hiera_tiny.pt: 156.0 MB


## Check

You should see videos in `data/` and the tiny checkpoint under `/workspace/segment-anything-2-real-time/checkpoints/`.

In [6]:
videos = sorted(SOURCE_VIDEO_DIRECTORY.glob("**/*.mp4"))
print(f"{len(videos)} video(s) in {SOURCE_VIDEO_DIRECTORY}")
for p in videos:
    print(" ", p.relative_to(SOURCE_VIDEO_DIRECTORY))

assert SAM2_DIR.exists(), f"SAM2 repo missing: {SAM2_DIR}"
assert CKPT.exists(), f"checkpoint missing: {CKPT}"
print("SAM2 OK")

10 video(s) in /workspace/Basketball-player-tracking/data
  boston-celtics-new-york-knicks-game-1-q1-01.54-01.48.mp4
  boston-celtics-new-york-knicks-game-1-q1-03.16-03.11.mp4
  boston-celtics-new-york-knicks-game-1-q1-04.28-04.20.mp4
  boston-celtics-new-york-knicks-game-1-q1-04.44-04.39.mp4
  boston-celtics-new-york-knicks-game-1-q1-05.13-05.09.mp4
  boston-celtics-new-york-knicks-game-1-q1-06.00-05.54.mp4
  boston-celtics-new-york-knicks-game-1-q1-07.41-07.34.mp4
  boston-celtics-new-york-knicks-game-1-q2-08.09-08.03.mp4
  boston-celtics-new-york-knicks-game-1-q2-08.43-08.38.mp4
  boston-celtics-new-york-knicks-game-1-q2-10.36-10.32.mp4
SAM2 OK
